# My data

In [1]:
import pandas as pd

In [7]:
import json

bad_lines = []
with open("base_questions/qa_simple.jsonl", "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            print(f"Line {line_num}: EMPTY (skipped)")
            continue
        try:
            parsed = json.loads(line)
            # Check if the parsed result is a dict (JSON object)
            if not isinstance(parsed, dict):
                print(f"Line {line_num}: Not a dict, it's a {type(parsed).__name__} -> {parsed[:50] if isinstance(parsed, str) else parsed}")
                bad_lines.append((line_num, line))
        except json.JSONDecodeError as e:
            print(f"Line {line_num}: INVALID JSON - {e}")
            print(f"Content: {repr(line[:100])}")  # Show first 100 chars
            bad_lines.append((line_num, line))

In [9]:
qa_simple = pd.read_json("base_questions/qa_simple.jsonl", lines=True)
qa_simple.head()

,prompt,answer,genre
0,What is the SI unit of force?,"The SI unit of force is the newton, defined as...",Science
1,What does the law of conservation of energy st...,It states that energy cannot be created or des...,Science
2,What is the difference between speed and veloc...,Speed is a scalar quantity measuring how fast ...,Science
3,What is the second law of thermodynamics often...,It states that the total entropy of an isolate...,Science
4,Which fundamental particle carries a positive ...,The proton carries a positive electric charge ...,Science


In [11]:
qa_advanced = pd.read_json("base_questions/qa_advanced.jsonl", lines=True)
qa_advanced.head()

,prompt,answer,genre
0,"If a car doubles its speed, how does its kinet...",Kinetic energy quadruples because it is propor...,Science
1,Why does ice float on liquid water despite bei...,Ice floats because its crystalline structure m...,Science
2,Compare the mechanisms of DNA replication in p...,Prokaryotes have a single origin of replicatio...,Science
3,How does a laser produce a coherent beam of li...,A laser uses stimulated emission where an inco...,Science
4,"Why do we see a rainbow after rain, and how do...",Rainbows form when sunlight is refracted and i...,Science


In [12]:
qa_instruction = pd.read_json("base_questions/qa_instruction.jsonl", lines=True)
qa_instruction.head()

,prompt,answer,genre
0,Explain the greenhouse effect in 3 bullet points.,- Solar radiation passes through the atmospher...,Science
1,Summarize the second law of thermodynamics in ...,The total entropy of an isolated system always...,Science
2,Give a step-by-step explanation of how a nucle...,"1. Neutrons strike uranium-235 nuclei, causing...",Science
3,Be concise and state the difference between ma...,"Mass is the amount of matter in an object, whi...",Science
4,Explain how a refrigerator works using exactly...,"- A refrigerant circulates through coils, evap...",Science


In [14]:
qa_edge = pd.read_json("base_questions/qa_edge.jsonl", lines=True)
qa_edge.head()

,prompt,answer,genre
0,What is the speed of the Earth?,Speed is relative; Earth's orbital speed aroun...,Science
1,How long does it take for an object to fall?,Without specifying height and ignoring air res...,Science
2,Does a heavier object fall faster than a light...,"In a vacuum, all objects fall at the same acce...",Science
3,Can a plane take off if it is on a conveyor be...,"Yes, because the wheels are free-spinning and ...",Science
4,What is the chemical formula for the newest el...,"As of 2026, no new elements have been official...",Science


In [15]:
qa_social = pd.read_json("base_questions/qa_social.jsonl", lines=True)
qa_social.head()

,prompt,answer,genre
0,What is the most polite way to interrupt a con...,"Say 'Excuse me' and wait for a pause, then bri...",social
1,How should you respond if a coworker takes cre...,"Calmly interject with a factual correction, su...",social
2,What is a good first step when you realize you...,"Apologize sincerely and directly, saying 'I'm ...",social
3,How can you politely decline an invitation to ...,"Say 'Thank you for the invitation, but I won't...",social
4,What should you do if you are in a group and s...,"When the interrupter pauses, turn to the inter...",social


In [16]:
qa_data = pd.concat([qa_simple, qa_advanced, qa_instruction, qa_edge, qa_social], ignore_index=True)
qa_data.sample(5)

,prompt,answer,genre
1828,"If your partner snores and it keeps you awake,...","Say 'I love you, but your snoring is affecting...",social
1406,Why does a bicycle stay upright when moving bu...,It stays upright due to gyroscopic precession ...,Science
1301,Explain why we have wisdom teeth in exactly 2 ...,- Our ancestors needed extra molars to chew to...,daily life explanations
635,"How does a fuse protect an electrical circuit,...",A fuse contains a wire that melts and breaks t...,Science
403,Why do we yawn when we are tired?,Yawning may help cool the brain and increase a...,daily life explanations


In [17]:
qa_data.describe()

,prompt,answer,genre
count,1862,1862,1862
unique,1857,1858,6
top,Why does a mirror reverse left and right but n...,The Sun.,Science
freq,2,2,450


In [19]:
qa_data.genre.value_counts()

genre
Science                    450
daily life explanations    351
reasoning questions        304
Programming                298
history                    280
social                     179
Name: count, dtype: int64

In [22]:
qa_data.to_json("base_questions/qa_full.jsonl", orient="records", lines=True, force_ascii=False)

# Alpaca

In [3]:
from datasets import load_dataset
import pandas as pd

In [2]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
len(dataset)

README.md: 0.00B [00:00, ?B/s]

e:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\areza\.cache\huggingface\hub\datasets--yahma--alpaca-cleaned. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better per

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

51760

In [4]:
dataset = dataset.filter(
    lambda x: x["input"] is None or x["input"].strip() == ""
)

print(f"Dataset size after filtering: {len(dataset):,}")

# Convert to pandas
df = dataset.to_pandas()

Filter:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset size after filtering: 32,603


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32603 entries, 0 to 32602
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   output       32603 non-null  object
 1   input        32603 non-null  object
 2   instruction  32603 non-null  object
dtypes: object(3)
memory usage: 764.3+ KB


In [9]:
for col in ["instruction", "output"]:
    df[f"{col}_chars"] = df[col].str.len()
    df[f"{col}_words"] = df[col].str.split().str.len()

In [10]:
df[[
    "instruction_chars",
    "output_chars"
]].describe()

,instruction_chars,output_chars
count,32603.000000,32603.000000
mean,62.842346,852.832377
std,44.563734,663.999887
min,9.000000,1.000000
25%,45.000000,233.000000
50%,57.000000,741.000000
75%,71.000000,1357.000000
max,2220.000000,4522.000000


In [11]:
df[[
    "instruction_words",
    "output_words"
]].describe()

,instruction_words,output_words
count,32603.000000,32603.000000
mean,10.706162,137.367942
std,7.666863,105.132681
min,2.000000,1.000000
25%,8.000000,40.000000
50%,10.000000,121.000000
75%,12.000000,216.000000
max,323.000000,507.000000


In [12]:
df.nlargest(5, "instruction_words")[
        ["instruction", "instruction_words"]
    ]

,instruction,instruction_words
73,You are a smart assistant designed to help hig...,323
320,"Given the following context, answer the questi...",295
56,Use the following pieces of context to answer ...,275
62,Use the following pieces of context to answer ...,274
427,You are a smart assistant designed to help hig...,250


In [13]:
df.nlargest(5, "output_words")[
        ["output", "output_words"]
    ]

,output,output_words
9945,Here is an implementation of a red-black tree ...,507
26907,Machine learning is a revolutionary branch of ...,450
31055,The coronavirus pandemic has made a significan...,450
24402,The departure of the United Kingdom (UK) from ...,448
29012,"In today’s digital age, it is extremely common...",447


In [17]:
def add_length(example):
    example["instruction_words"] = len(example["instruction"].split())
    example["output_words"] = len(example["output"].split())
    return example

dataset = dataset.map(add_length)

Map:   0%|          | 0/32603 [00:00<?, ? examples/s]

In [21]:
test = dataset.filter(
    lambda x: x["instruction_words"] <= 30
    and x["output_words"] <= 130
)
len(test)

Filter:   0%|          | 0/26889 [00:00<?, ? examples/s]

17071